# Kaggle version: MoE classification head experiments on ViHSD/ViANLI.

In [ ]:
from pathlib import Path
import os

# Dùng repo đã clone/upload; chọn đường dẫn phù hợp phiên làm việc.
REPO_DIR = Path("/kaggle/working/code")

if Path("train.py").exists():
    REPO_DIR = Path.cwd()

REPOSITORY_URL = "https://github.com/lngphgthao/moe-vihsd.git"
BRANCH = "main" 

if not REPO_DIR.exists():
    !git clone --depth 1 --branch "$BRANCH" "$REPOSITORY_URL" "$REPO_DIR"
    
assert (REPO_DIR / "train.py").is_file(), "Đặt REPO_DIR đến checkout/upload chứa code mới"
os.chdir(REPO_DIR)

In [ ]:
!python -m pip install -q -r requirements.txt
assert _exit_code == 0, "Cài dependency thất bại"

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

for key in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = secrets.get_secret(key)
    except Exception:
        print(f"Chưa lấy được secret {key}; thêm vào Kaggle Secrets nếu cần")
        
os.environ["RUNS_DIR"] = "/kaggle/working/runs"

## Chọn experiment configs

- CONFIG_PATH: path cho experiment config riêng (exp-010/exp-011).
- GROUP CONFIG_PATH: path cho experiment config run_group

In [ ]:
# CONFIG_PATH = "configs/experiments/vianli/exp-010.yaml"

# GROUP_CONFIG_PATH = "configs/groups/phobert_original_pair_baseline.yaml"

## Smoke — chạy đúng một option

Bỏ comment lệnh và assert của **A** (hai config riêng) hoặc **B** (`run_group`),
giữ option còn lại comment. Smoke dùng 1 epoch/64 train/32 validation, W&B tắt;
không dùng metric smoke để chọn model. Không Run All trước khi chọn option.

In [ ]:
# OPTION A — chạy riêng từng config:
# !python train.py --config {CONFIG_PATH} --smoke-test --set logging.use_wandb=false
# assert _exit_code == 0, "Smoke thất bại; không chạy cell full"

In [ ]:
# OPTION B — chạy cùng hai config qua run_group:
# !python scripts/run_group.py --group-config "{GROUP_CONFIG_PATH}" --smoke-test --set logging.use_wandb=false --auto-run-ids
# assert _exit_code == 0, "Group smoke thất bại; không chạy cell full"

## Full baseline — chỉ sau khi cả hai smoke đạt

Kiểm tra status, loss hữu hạn, checkpoint và dữ liệu trước khi chạy. Chọn đúng một option
như cell smoke; hai config chạy tuần tự trên GPU 0, seed 42. Group chính có ID cố định
run-022/025. Đối chiếu tracker/W&B/storage; nếu đã dùng, sửa group và lệnh A cho khớp.


In [ ]:
assert os.environ.get("WANDB_API_KEY"), "Thêm WANDB_API_KEY trước full run"

In [ ]:
# OPTION A — chạy riêng từng config:
# !python train.py --config {CONFIG_PATH} --no-smoke-test
# assert _exit_code == 0, "Full thất bại; giữ artifact/log"

In [ ]:
# OPTION B — chạy cùng hai config qua run_group:
# !python scripts/run_group.py --group-config {GROUP_CONFIG_PATH} --no-smoke-test --auto-run-ids
# assert _exit_code == 0, "Group full thất bại; giữ artifact/log"

## Aggregate và lưu tracker

Aggregate chỉ báo cáo full run completed; smoke vẫn có trong `runs.csv` nhưng không
được gộp vào thống kê full. Giữ run directories và tracker khi kết thúc phiên.

In [ ]:
!python scripts/aggregate_results.py --runs-dir /kaggle/working/runs --output-dir /kaggle/working/runs/analysis
assert _exit_code == 0, "Tổng hợp kết quả thất bại"

!cp experiments/experiment_tracker.csv /kaggle/working/experiment_tracker.csv
assert _exit_code == 0, "Lưu tracker thất bại"

## Test set — chỉ sau khi khóa finalist

Không chạy đoạn mẫu dưới đây cho baseline phát triển. Chỉ bỏ comment sau khi đã có
finalist config khóa và run đầy đủ đúng seed/provenance.

In [ ]:
# !python evaluate.py --run-dir "/kaggle/working/runs/<run-id>" --finalist-config "configs/finalists/<dataset>/<experiment-id>.yaml" --final-evaluation
# assert _exit_code == 0, "Final evaluation thất bại; giữ log và artifact"